In [1]:
import os
import glob
import math
import pandas as pd
import numpy as np
import ee

import yaml

# Load Configuration
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

# 1. Initialize GEE
ee.Authenticate()
ee.Initialize(project=config['gee']['project_id'], opt_url='https://earthengine-highvolume.googleapis.com')
print("GEE Initialized successfully.")

# # 2. Define Paths
# INPUT_DIR = "./shc_data/AEZS/AGRI_2023-24/"
# GDRIVE_FOLDER = "GEE_LULC_All_AEZs_ratinder"

# # 3. Load the LULC Asset
# lulc_path = "projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_2023_2024"
# lulc_image = ee.Image(lulc_path).select('predicted_label')

# 2. Define Paths and Parameters from Config
INPUT_DIR = config['paths']['aez_data_dir']
DOWNLOAD_DIR = config['paths']['lulc_download_dir']
GDRIVE_FOLDER = config['gdrive_exports']['lulc_folder']

CHUNK_SIZE = 5000
CROP_LABELS = config['parameters']['crop_labels']

# 3. Load the LULC Asset
lulc_path = config['gee']['lulc_asset_path']
lulc_image = ee.Image(lulc_path).select('predicted_label')

print(f"Reading AEZ data from: {INPUT_DIR}")
print(f"Exporting to GDrive folder: {GDRIVE_FOLDER}")


/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


GEE Initialized successfully.


In [2]:
# Get all AEZ files (assuming they follow the pattern AEZ_1_tagged.csv, AEZ_2_tagged.csv, etc.)
aez_files = glob.glob(os.path.join(INPUT_DIR, "AEZ_*_tagged.csv"))
# Filter out any files that might already have "LULC" or "Filtered" in the name just in case
aez_files = [f for f in aez_files if "LULC" not in f and "Filtered" not in f and "Clean" not in f]

print(f"Found {len(aez_files)} AEZ files to process.")

for file_path in aez_files:
    aez_name = os.path.basename(file_path).replace('_tagged.csv', '')
    print(f"\n--- Processing {aez_name} ---")
    
    # Load and prep index
    df = pd.read_csv(file_path)
    df = df.reset_index().rename(columns={'index': 'orig_idx'})
    
    num_chunks = int(np.ceil(len(df) / CHUNK_SIZE))
    print(f"Submitting {num_chunks} tasks...")

    for i in range(num_chunks):
        start = i * CHUNK_SIZE
        end = min((i + 1) * CHUNK_SIZE, len(df))
        
        # Strip to essential columns only for payload efficiency
        chunk_df = df.iloc[start:end][['latitude', 'longitude', 'orig_idx']]
        
        # Use itertuples() for massive speedup over iterrows()
        features = [
            ee.Feature(ee.Geometry.Point([row.longitude, row.latitude]), {'orig_idx': int(row.orig_idx)}) 
            for row in chunk_df.itertuples(index=False)
        ]
        
        fc = ee.FeatureCollection(features)
        
        # Sample labels at 10m resolution
        sampled_fc = lulc_image.sampleRegions(
            collection=fc,
            scale=10,
            geometries=False
        )
        
        task_name = f'{aez_name}_LULC_Part_{i+1}'
        task = ee.batch.Export.table.toDrive(
            collection=sampled_fc,
            description=task_name,
            fileFormat='CSV',
            folder=GDRIVE_FOLDER,
            fileNamePrefix=task_name
        )
        task.start()
        
    print(f"All tasks for {aez_name} submitted.")

print("\n✅ All AEZ tasks active. Monitor at: https://code.earthengine.google.com/tasks")

Found 19 AEZ files to process.

--- Processing AEZ_20 ---
Submitting 1 tasks...
All tasks for AEZ_20 submitted.

--- Processing AEZ_19 ---
Submitting 6 tasks...
All tasks for AEZ_19 submitted.

--- Processing AEZ_18 ---
Submitting 15 tasks...
All tasks for AEZ_18 submitted.

--- Processing AEZ_8 ---
Submitting 39 tasks...
All tasks for AEZ_8 submitted.

--- Processing AEZ_9 ---
Submitting 45 tasks...
All tasks for AEZ_9 submitted.

--- Processing AEZ_16 ---
Submitting 5 tasks...
All tasks for AEZ_16 submitted.

--- Processing AEZ_4 ---
Submitting 82 tasks...
All tasks for AEZ_4 submitted.

--- Processing AEZ_5 ---
Submitting 30 tasks...
All tasks for AEZ_5 submitted.

--- Processing AEZ_17 ---
Submitting 5 tasks...
All tasks for AEZ_17 submitted.

--- Processing AEZ_15 ---
Submitting 72 tasks...
All tasks for AEZ_15 submitted.

--- Processing AEZ_7 ---
Submitting 5 tasks...
All tasks for AEZ_7 submitted.

--- Processing AEZ_6 ---
Submitting 28 tasks...
All tasks for AEZ_6 submitted.

-

In [4]:
import pandas as pd
import glob
import os


# Get AEZ files again
aez_files = glob.glob(os.path.join(INPUT_DIR, "AEZ_*_tagged.csv"))
aez_files = [f for f in aez_files if "LULC" not in f and "Filtered" not in f and "Clean" not in f]

for file_path in aez_files:
    aez_name = os.path.basename(file_path).replace('_tagged.csv', '')
    
    # 1. Find all downloaded chunks for this specific AEZ
    chunk_files = glob.glob(os.path.join(DOWNLOAD_DIR, f"{aez_name}_LULC_Part_*.csv"))
    
    if not chunk_files:
        print(f"Skipping {aez_name}: No downloaded chunks found in {DOWNLOAD_DIR}")
        continue
        
    # 2. Robustly load chunks (skipping empty files)
    valid_dfs = []
    for f in chunk_files:
        try:
            if os.path.getsize(f) > 0:
                temp_df = pd.read_csv(f)
                if not temp_df.empty:
                    valid_dfs.append(temp_df)
        except pd.errors.EmptyDataError:
            pass # Silently skip empty files
            
    if not valid_dfs:
        print(f"Skipping {aez_name}: All chunk files were empty.")
        continue

    # Combine valid chunks
    df_labels = pd.concat(valid_dfs, ignore_index=True)
    df_labels = df_labels[['orig_idx', 'predicted_label']]
    
    # 3. Load original AEZ file and set up index exactly as before
    df_original = pd.read_csv(file_path)
    df_original = df_original.reset_index().rename(columns={'index': 'orig_idx'})
    
    # 4. Merge robustly on orig_idx
    df_merged = pd.merge(df_original, df_labels, on='orig_idx', how='inner')
    
    # 5. Filter for target crop labels
    df_filtered = df_merged[df_merged['predicted_label'].isin(CROP_LABELS)].copy()
    
    # 6. Clean up columns (drop orig_idx and predicted_label to match original format)
    df_filtered = df_filtered.drop(columns=['orig_idx', 'predicted_label'])
    
    # 7. Save output
    output_path = os.path.join(INPUT_DIR, f"{aez_name}_LULC_Filtered.csv")
    df_filtered.to_csv(output_path, index=False)
    
    print(f"--- {aez_name} ---")
    print(f"Original: {len(df_original)} points")
    print(f"Filtered: {len(df_filtered)} agricultural points")
    print(f"Removed:  {len(df_original) - len(df_filtered)} noise points\n")

print("✅ All AEZs successfully merged and filtered. Ready for additional feature fetching!")

Skipping AEZ_20: All chunk files were empty.
--- AEZ_19 ---
Original: 27938 points
Filtered: 7105 agricultural points
Removed:  20833 noise points

--- AEZ_18 ---
Original: 73301 points
Filtered: 36205 agricultural points
Removed:  37096 noise points

--- AEZ_8 ---
Original: 193102 points
Filtered: 105113 agricultural points
Removed:  87989 noise points

--- AEZ_9 ---
Original: 221566 points
Filtered: 127825 agricultural points
Removed:  93741 noise points

--- AEZ_16 ---
Original: 21207 points
Filtered: 3403 agricultural points
Removed:  17804 noise points

--- AEZ_4 ---
Original: 406655 points
Filtered: 263778 agricultural points
Removed:  142877 noise points

--- AEZ_5 ---
Original: 145078 points
Filtered: 71613 agricultural points
Removed:  73465 noise points

--- AEZ_17 ---
Original: 20745 points
Filtered: 3569 agricultural points
Removed:  17176 noise points

--- AEZ_15 ---
Original: 357019 points
Filtered: 148020 agricultural points
Removed:  208999 noise points

--- AEZ_7 ---
O